In [ ]:
import sys
from enum import Enum
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent          # one level up
SRC = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC))


class Server(str, Enum):
    JETSON = 'http://jetson-edge:8000/start'
    LOCAL = 'http://localhost:8000/start'

    def __str__(self):
        return self.value

In [ ]:
import requests
from uuid import uuid4
from IPython.display import Markdown

API_URL = Server.JETSON
thread_id = str(uuid4())
def main():
    print("Started Session. 'exit' or Strg+C to end.")
    
    while True:
        try:
            user_question = input("You: ").strip()
            if user_question.lower() in {"exit", "quit", ""}:
                print("End Session...")
                break

            response = requests.post(
                API_URL,
                json={
                    "user_question": user_question,
                    "thread_id": thread_id
                },
                timeout=60
            )

            if response.status_code == 200:
                data = response.json()
                result = data.get("result")

                if isinstance(result, dict) and "final_answer" in result:
                    final = result["final_answer"]
                    
                    if isinstance(final, dict):
                        output = final.get("content", final)
                    else:
                        output = final
                else:
                    output = "No final_answer found"

                #print("Gibson:", pprint.pformat(output, width=80))
                display(Markdown(f"**You:** {user_question}"))
                display(Markdown(f"**Gibson:** {output}"))


            else:
                print(f"Error: {response.status_code} - {response.text}")

        except KeyboardInterrupt:
            print("\nSession ended.")
            break
        except Exception as e:
            print(f"Error: {e}")

if __name__ == "__main__":
    main()


In [ ]:
import os
os.getcwd()